# 03 — Modelling & Evaluation

Supervised regression pipelines with preprocessing baked in.

**Models**
- Linear Regression, Ridge
- K-Nearest Neighbours
- Random Forest
- Gradient Boosting
- XGBoost, LightGBM

Hyper-parameters selected by GridSearchCV (R², 3-fold).  
Hold-out test set (20 %) is used for the final comparison.


In [19]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

results = pd.read_csv(ROOT / "models" / "model_comparison.csv")
meta = json.loads((ROOT / "models" / "meta.json").read_text())
print("Best model:", meta["best_model"], f"(R² = {meta['best_test_r2']:.4f})")
results[["model", "r2", "rmse", "mae", "cv_r2"]]

Best model: GradientBoosting (R² = 0.7225)


,model,r2,rmse,mae,cv_r2
0,XGBoost,0.722932,0.946100,0.744682,0.727408
1,GradientBoosting,0.721694,0.948211,0.746545,0.729008
2,LightGBM,0.720959,0.949464,0.748642,0.725782
3,RandomForest,0.705550,0.975325,0.773274,0.714949
4,KNN,0.675134,1.024463,0.817943,0.684575
5,Ridge,0.672520,1.028575,0.824500,0.685156
6,LinearRegression,0.672512,1.028589,0.824487,NaN


## Model ranking by test R²

In [20]:
fig = px.bar(results.sort_values("r2"), x="r2", y="model", orientation="h",
             text=results.sort_values("r2")["r2"].round(3),
             title="Test-set R² by model", labels={"r2": "R²", "model": ""},
             color="r2", color_continuous_scale="Teal")
fig.update_layout(template="plotly_white", height=400, coloraxis_showscale=False)
fig.show()

## RMSE comparison

In [21]:
fig = px.bar(results.sort_values("rmse", ascending=False), x="rmse", y="model",
             orientation="h", title="Test-set RMSE (lower is better)",
             labels={"rmse": "RMSE", "model": ""}, color="rmse",
             color_continuous_scale="OrRd")
fig.update_layout(template="plotly_white", height=400, coloraxis_showscale=False)
fig.show()

## Pipeline structure

Every model is wrapped as:

```
Pipeline([
    ("preprocess", ColumnTransformer(
        numeric     → median impute + StandardScaler
        boolean     → most-frequent impute
        categorical → constant "Other" impute + OneHotEncoder
    )),
    ("model", <estimator>)
])
```

Missing job types or platforms are treated as **Other** (option not listed on the survey form).
This keeps train and inference transformations identical and avoids leakage.


In [22]:
model = joblib.load(ROOT / "models" / "best_model.joblib")
print(type(model))
print(model)

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('sc',
                                                                   StandardScaler())]),
                                                  ['age',
                                                   'daily_social_media_time',
                                                   'number_of_notifications',
                                                   'work_hours_per_day',
                                                   'perceived_productivity_score',
                                                   'stress_level',
                                                   'sleep_hours',
                                    

## Residual check on the hold-out set

In [23]:
from src.preprocess import clean_frame, split_xy
from sklearn.model_selection import train_test_split

df = pd.read_csv(ROOT / "data" / "cleaned.csv")
X, y = split_xy(df)
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_pred = model.predict(X_test)

resid = y_test - y_pred
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_pred, y=resid, mode="markers", opacity=0.35,
                         marker=dict(size=4, color="#3b82f6")))
fig.add_hline(y=0, line_dash="dash", line_color="#ef4444")
fig.update_layout(template="plotly_white", height=400,
                  title="Residuals vs predicted",
                  xaxis_title="Predicted", yaxis_title="Residual")
fig.show()
print("Residual mean:", round(resid.mean(), 4))
print("Residual std :", round(resid.std(), 4))

Residual mean: -0.0218
Residual std : 0.9466


## Predicted vs actual

In [24]:
fig = px.scatter(x=y_test, y=y_pred, opacity=0.3,
                 labels={"x": "Actual", "y": "Predicted"},
                 title="Predicted vs actual productivity")
fig.add_trace(go.Scatter(x=[3, 10], y=[3, 10], mode="lines",
                         line=dict(color="red", dash="dash"), name="Ideal"))
fig.update_layout(template="plotly_white", height=420)
fig.show()